# Module 5 — From Prompts to Agents

**NetOps Co. · CELL-031A · ~45 minutes**

Everything so far has been one call, one answer. An agent is different: **its next action depends
on the result of its last one**, decided by the model rather than hard-coded by you.

We build the smallest possible version by hand — no framework — so the mechanics stay visible.
When the libraries change next year, this will still make sense.

## Setup — about 60 seconds

Run this once per session. Colab gives you a fresh machine each time, so the clone and
install have to happen again — that is normal, not a mistake.

**Your API key.** Click the 🔑 key icon in the left sidebar, add a secret named
`GEMINI_API_KEY`, and toggle *Notebook access* on. Get a free key at
[aistudio.google.com](https://aistudio.google.com) — no credit card.

Never paste a key into a cell. Notebooks get shared, and the key goes with them.

**No key, no run.** Every cell below calls a model, and this module has no offline mode: the lab *is* the model deciding what to do next, so a canned version would be a slideshow.

In [ ]:
!git clone -q https://github.com/telcobytes/netops-genai-course.git 2>/dev/null || (cd netops-genai-course && git pull -q)
!pip install -q google-genai pydantic

import sys, os

# Repo root, resolved rather than hardcoded: the clone Colab just made, the
# checkout this notebook lives in, or the directory it was launched from.
REPO_ROOT = next((p for p in ('/content/netops-genai-course',
                              os.path.abspath(os.path.join(os.getcwd(), '..')),
                              os.getcwd())
                  if os.path.isdir(os.path.join(p, 'data'))), None)
assert REPO_ROOT, 'Could not find the repo root. Run this notebook from inside the checkout.'
sys.path.insert(0, os.path.join(REPO_ROOT, 'data'))

# Key from Colab secrets, with a local fallback so this notebook also runs
# in plain Jupyter.
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    if not os.environ.get('GEMINI_API_KEY'):
        import getpass
        os.environ['GEMINI_API_KEY'] = getpass.getpass('GEMINI_API_KEY: ')

print('Key loaded:', bool(os.environ.get('GEMINI_API_KEY')))
print('Module 5 — ReAct loop')

### Sanity check — no API key needed

`mock_tools.py` is pure standard library. If this prints numbers, your environment is
working and the rest of the notebook will run.

In [ ]:
import mock_tools
summary = mock_tools.get_cell_kpis('CELL-031A')
print('window     :', summary['window_start'], '->', summary['window_end'])
print('samples    :', summary['sample_count'])
print('rolling avg:', summary['rolling_avg'])
for c in summary['thresholds_crossed']:
    print(f"  CROSSED  {c['metric']} = {c['value']} ({c['comparison']} {c['threshold']})")

---
## 1. The loop: perceive → reason → act → observe

Run the real agent from the repo and watch it work. It gets three read-only tools and a question.

In [ ]:
sys.path.insert(0, os.path.join(REPO_ROOT, 'module05-react-loop'))
import io, contextlib
from react_agent import run_react_agent

# One run, captured. The next cell re-uses this log instead of paying for a second run.
# `session` is the agent's memory; we use it for the follow-up two cells down.
session = []
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    answer = run_react_agent('Why is CELL-031A underperforming right now?', history=session)
log = buf.getvalue()

print(log)
print('ANSWER:', answer)

---
## 2. The same run, rendered as a trace

Text scrolls past. A trace you can read is how you actually debug an agent — and it is exactly
what Module 10 turns into tracing and evaluation.

Notice the shape of the reasoning: it checks KPIs, then topology, *then* concludes. That order
is not an accident — it comes from the 4-layer diagnostic order in the system prompt.

Notice the step lines too: `--- Step 3 --- (sending 2,461 chars in 6 messages)`. Every step
re-sends the whole conversation, so the cost of a run grows with steps × history, not with steps.
That is why the loop trims the raw `readings` out of each KPI observation before storing it —
measured, that one field is 31% of everything this run sends.

In [ ]:
import re, nb_viz
from IPython.display import HTML, display

steps = []
for block in re.split(r'(?=Thought:)', log):        # `log` came from the run above
    t = re.search(r'Thought:\s*(.+)', block)
    a = re.search(r'Action:\s*(.+)', block)
    o = re.search(r'Observation:\s*(.+)', block)
    if t or a:
        obs = o.group(1).strip() if o else ''
        steps.append({'thought': t.group(1).strip() if t else '',
                      'action':  a.group(1).strip() if a else '',
                      # observations are JSON; show the head, not 900 characters of it
                      'observation': (obs[:180] + ' …') if len(obs) > 180 else obs})
display(HTML(nb_viz.trace(steps, caption='ReAct trace — CELL-031A')))


---
## 3. Memory: ask a follow-up

`session` holds the transcript of the run above — the agent's only memory, and it lives in your
list, not in the model. Hand it back with a new question and the agent continues instead of
starting over.

Watch the step count. Measured: the first question took 5 steps, the follow-up
took 2 — it already had the KPIs, the topology and the alarms, and spent its one tool call
checking a neighbour's headroom before recommending where to offload. Memory does not make the
agent skip work; it stops it repeating work it already did.


In [ ]:

print('session carries', len(session), 'messages into the follow-up\n')
print(run_react_agent('What should the NOC do about it first?', history=session))


---
## 3. The circuit breakers — and what happens without them

Two guards in that loop look like defensive plumbing. They are not optional.

- **`max_steps`** — a hard ceiling on how many times round the loop
- **`call_history`** — cycle detection on `(tool, argument)` pairs

Squeeze the budget and watch the agent hit the wall mid-diagnosis.

In [ ]:
print(run_react_agent('Why is CELL-031A underperforming this morning?', max_steps=2))

> It stops without concluding. That is the *correct* behaviour — an agent that knows it ran out
> of budget is far safer than one that keeps going.

> The budget is not only about the bill. Every step appends another observation to the context, and
> those observations are large — one `get_cell_kpis` result is about 1,800 characters of raw counters.
> A loop with no ceiling grows its own prompt until the answer degrades, which is what Module 1
> showed you about long windows. It never errors. It just gets slower, more expensive and worse.

---
## Your turn

1. **Ask about CELL-022A.** No KPI threshold is crossed there, but SITE-022 does have one MINOR
   VoLTE alarm described as *within tolerance*. Does the agent say "nothing wrong", or does it
   report the minor alarm without escalating it? The second is the answer you want.
2. **Take a tool away.** `react_agent.TOOLS.pop('lookup_topology')` removes it from the dispatch
   table — note the system prompt still advertises it, so the model asks for it anyway and the
   harness answers `Unknown tool`. Watch what it concludes without neighbour data. This is the
   failure the whole capstone is built to prevent.
3. **Ask for something it cannot do.** `create_ticket` is not in `TOOLS`. Ask the agent to open a
   ticket and watch the harness refuse it — the model can propose anything; your code decides.
4. **Squeeze and stretch the budget.** `max_steps=2` stops it mid-diagnosis; `max_steps=10` gives
   it room it does not need. Which number would you ship, and what would you measure to defend it?
5. **Watch it recover from a bad argument.** Ask it about `SITE-031` as if it were a cell —
   `run_react_agent('Why is SITE-031 underperforming right now?')`. Sooner or later it passes an
   id to the wrong tool and the observation comes back as
   `unknown site_id 'CELL-031A' — known sites: ...`. Watch the next step fix itself. A tool that
   returns `{}` would have taught it nothing.
6. **Run cell 1 twice.** Same question, same data, different number of steps — 4 and 5 when this
   was measured. Agent runs are not reproducible, which is the problem Module 10 exists to solve.

In [ ]:
# Your turn — 1
print(run_react_agent('Is there anything wrong with CELL-022A?'))

# 2 — uncomment to run without topology (one run, ~4 calls)
# import react_agent
# react_agent.TOOLS.pop('lookup_topology', None)
# print(run_react_agent('Why is CELL-031A underperforming right now?'))

---
**Next — and this is where the notebooks stop.**

From Module 6 on we are writing files, not cells. Not because notebooks stopped working, but
because the lesson changes: tool schemas, a packaged Skill on disk, an MCP server running as a
process, an eval suite in CI. Those *are* files, and putting them in a notebook would teach the
opposite of the point.

Clone the repo locally and open Module 6 in your editor:

```bash
git clone https://github.com/telcobytes/netops-genai-course.git
cd netops-genai-course && pip install -r requirements.txt
python module06-noc-assistant/noc_assistant.py
```

You have outgrown the notebook. That is progress, not friction.